# 주요 개선 내용
1. LightGBM + Optuna 파라미터 튜닝
2. LightGBM + XGBoost 앙상블

In [1]:
import pandas as pd
import numpy as np
import random, os, warnings
warnings.filterwarnings('ignore')

import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

import xgboost as xgb
from lightgbm import LGBMRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt

In [2]:
# 시드 고정
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(42)


In [3]:
# 데이터 로드
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
print(f"Train: {train.shape} | Test: {test.shape}")

Train: (7500, 11) | Test: (7500, 10)


In [4]:
# 피처 엔지니어링
# Weight 관련은 EDA에서 의미없음 확인 → 제거
# Duration, BPM, Temp 중심 파생변수
def feature_engineering(df):
    df = df.copy()

    # Height 통합
    df['Height_Total_Inches'] = df['Height(Feet)'] * 12 + df['Height(Remainder_Inches)']

    # Temp_diff: 운동에 의한 체온 상승폭 (핵심)
    df['Temp_diff'] = df['Body_Temperature(F)'] - 98.6

    # Duration 구간화
    df['Duration_bin'] = pd.cut(
        df['Exercise_Duration'],
        bins=[0, 10, 20, 30],
        labels=[0, 1, 2]
    ).astype(int)

    # ── Duration 중심 Interaction (EDA 상관계수 0.95+) ──
    df['Duration_x_BPM'] = df['Exercise_Duration'] * df['BPM']
    df['Duration_x_TempDiff'] = df['Exercise_Duration'] * df['Temp_diff']
    df['BPM_x_TempDiff'] = df['BPM'] * df['Temp_diff']

    # 다항식 (비선형 관계 보강)
    df['Duration_sq'] = df['Exercise_Duration'] ** 2
    df['Temp_diff_sq'] = df['Temp_diff'] ** 2

    # 3-way interaction
    df['Dur_BPM_TempDiff'] = df['Exercise_Duration'] * df['BPM'] * df['Temp_diff']

    # Gender 인코딩
    df['Gender_encoded'] = (df['Gender'] == 'M').astype(int)

    return df

train_fe = feature_engineering(train)
test_fe = feature_engineering(test)

# ── 피처 선택: Weight 관련 제거, Duration/BPM/Temp 중심 ──
feature_cols = [
    'Exercise_Duration',
    'Body_Temperature(F)',
    'BPM',
    'Height_Total_Inches',
    'Age',
    'Temp_diff',
    'Duration_bin',
    'Duration_x_BPM',
    'Duration_x_TempDiff',
    'BPM_x_TempDiff',
    'Duration_sq',
    'Temp_diff_sq',
    'Dur_BPM_TempDiff',
    'Gender_encoded',
]

X = train_fe[feature_cols]
y = train_fe['Calories_Burned']
X_test = test_fe[feature_cols]

# 타겟 로그 변환
y_log = np.log1p(y)

print(f"\n피처 수: {len(feature_cols)}")
print(f"타겟 변환: log1p (원본 평균={y.mean():.1f}, 변환 후={y_log.mean():.3f})")



피처 수: 14
타겟 변환: log1p (원본 평균=89.4, 변환 후=4.155)


In [10]:
# K-Fold 설정
N_SPLITS = 5
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

def evaluate_cv(model_class, params, X, y_log, y_orig):
    """5-Fold CV 수행, log→원본 복원 후 RMSE 계산"""
    fold_rmse = []
    models = []

    for train_idx, val_idx in kf.split(X):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val_log = y_log.iloc[train_idx], y_log.iloc[val_idx]
        y_val_orig = y_orig.iloc[val_idx]

        model = model_class(**params)
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val_log)], verbose=False)

        pred_log = model.predict(X_val)
        pred_orig = np.expm1(pred_log)  # log1p 역변환
        pred_orig = np.clip(pred_orig, 0, None)  # 음수 방지

        rmse = np.sqrt(mean_squared_error(y_val_orig, pred_orig))
        fold_rmse.append(rmse)
        models.append(model)

    return fold_rmse, models

In [13]:
# LightGBM + Optuna

print("\n" + "=" * 70)
print("  STEP 1: LightGBM + Optuna 최적화")
print("=" * 70)

def lgb_objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 500, 3000),
        'max_depth':        trial.suggest_int('max_depth', 3, 10),
        'learning_rate':    trial.suggest_float('learning_rate', 0.005, 0.2, log=True),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_float('min_child_weight', 0.1, 20.0),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'num_leaves':       trial.suggest_int('num_leaves', 15, 127),
        'min_split_gain':   trial.suggest_float('min_split_gain', 0.0, 1.0),
        'random_state': 42,
        'n_jobs': -1,
        'verbosity': -1,
    }

    fold_rmse = []
    for train_idx, val_idx in kf.split(X):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val_log = y_log.iloc[train_idx], y_log.iloc[val_idx]
        y_val_orig = y.iloc[val_idx]

        model = LGBMRegressor(**params)
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val_log)])

        pred_log = model.predict(X_val)
        pred_orig = np.expm1(pred_log)
        pred_orig = np.clip(pred_orig, 0, None)

        rmse = np.sqrt(mean_squared_error(y_val_orig, pred_orig))
        fold_rmse.append(rmse)

    return np.mean(fold_rmse)

lgb_study = optuna.create_study(
    direction='minimize',
    sampler=TPESampler(seed=42),
    study_name='lgb_calories'
)
lgb_study.optimize(lgb_objective, n_trials=100, show_progress_bar=True)

lgb_best = lgb_study.best_params
lgb_best.update({'random_state': 42, 'n_jobs': -1, 'verbosity': -1})

print(f"\n  LightGBM Best CV RMSE: {lgb_study.best_value:.6f}")



  STEP 1: LightGBM + Optuna 최적화


Best trial: 89. Best value: 2.17984: 100%|██████████| 100/100 [15:51<00:00,  9.51s/it]


  LightGBM Best CV RMSE: 2.179841

  STEP 2: XGBoost + Optuna 최적화 (log1p 타겟)


In [ ]:

# XGBoost + Optuna (with log1p target)
print("\n" + "=" * 70)
print("  STEP 2: XGBoost + Optuna 최적화 (log1p 타겟)")
print("=" * 70)


def xgb_objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 500, 3000),
        'max_depth':        trial.suggest_int('max_depth', 3, 10),
        'learning_rate':    trial.suggest_float('learning_rate', 0.005, 0.2, log=True),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'gamma':            trial.suggest_float('gamma', 0.0, 5.0),
        'random_state': 42,
        'n_jobs': -1,
        'tree_method': 'hist',
        'verbosity': 0,
    }

    fold_rmse = []
    for train_idx, val_idx in kf.split(X):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val_log = y_log.iloc[train_idx], y_log.iloc[val_idx]
        y_val_orig = y.iloc[val_idx]

        model = xgb.XGBRegressor(**params)
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val_log)], verbose=False)

        pred_log = model.predict(X_val)
        pred_orig = np.expm1(pred_log)
        pred_orig = np.clip(pred_orig, 0, None)

        rmse = np.sqrt(mean_squared_error(y_val_orig, pred_orig))
        fold_rmse.append(rmse)

    return np.mean(fold_rmse)

xgb_study = optuna.create_study(
    direction='minimize',
    sampler=TPESampler(seed=42),
    study_name='xgb_calories'
)
xgb_study.optimize(xgb_objective, n_trials=100, show_progress_bar=True)

xgb_best = xgb_study.best_params
xgb_best.update({'random_state': 42, 'n_jobs': -1, 'tree_method': 'hist', 'verbosity': 0})

print(f"\n  XGBoost Best CV RMSE: {xgb_study.best_value:.6f}")


  STEP 1: LightGBM + Optuna 최적화


  0%|          | 0/100 [00:00<?, ?it/s]

[W 2026-02-12 21:29:46,805] Trial 0 failed with parameters: {'n_estimators': 1436, 'max_depth': 10, 'learning_rate': 0.07441632389160634, 'subsample': 0.8394633936788146, 'colsample_bytree': 0.5780093202212182, 'min_child_weight': 3.2042909546904323, 'reg_alpha': 3.3323645788192616e-08, 'reg_lambda': 0.6245760287469887, 'num_leaves': 82, 'min_split_gain': 0.7080725777960455} because of the following error: TypeError("LGBMRegressor.fit() got an unexpected keyword argument 'verbose'").
Traceback (most recent call last):
  File "/Users/admin/Desktop/DB/venv/lib/python3.13/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "/var/folders/zd/8r845kcj6nn1ym5f96t9hpw00000gn/T/ipykernel_15149/2028329756.py", line 58, in lgb_objective
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val_log)], verbose=False)
    ~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: LGBMRegressor.fit() got an unexpected keyword argument

TypeError: LGBMRegressor.fit() got an unexpected keyword argument 'verbose'